In [2]:
%pip install pandas matplotlib ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install --upgrade ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# --------------------------------------------------
# 1. LOAD THE DATASETS
# --------------------------------------------------

# Change these paths to match the location of your CSV files.
FILES = {
    "UCDP/PRIO Armed Conflict": "UcdpPrioConflict_v26_1.csv",
    "UCDP GED Events": "GEDEvent_v26_1.csv"
}

datasets = {}

for name, filepath in FILES.items():
    try:
        datasets[name] = pd.read_csv(
            filepath,
            low_memory=False,
            encoding="utf-8"
        )
        print(f"Loaded: {name} — {datasets[name].shape[0]:,} rows, "
              f"{datasets[name].shape[1]} columns")

    except FileNotFoundError:
        print(f"File not found: {filepath}")
    except Exception as e:
        print(f"Error loading {filepath}: {e}")


# --------------------------------------------------
# 2. CREATE THE INTERACTIVE VIEWER
# --------------------------------------------------

if not datasets:
    raise ValueError("No datasets were loaded. Check your CSV file paths.")

dataset_selector = widgets.Dropdown(
    options=list(datasets.keys()),
    description="Dataset:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

column_selector = widgets.SelectMultiple(
    options=[],
    description="Columns:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px", height="260px")
)

row_limit = widgets.IntSlider(
    value=20,
    min=5,
    max=200,
    step=5,
    description="Rows to show:",
    style={"description_width": "initial"},
    continuous_update=False,
    layout=widgets.Layout(width="450px")
)

start_row = widgets.IntText(
    value=0,
    description="Start at row:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

show_all_rows = widgets.Checkbox(
    value=False,
    description="Show all rows (may be slow)"
)

show_metadata = widgets.Checkbox(
    value=True,
    description="Show column data types and missing values"
)

display_button = widgets.Button(
    description="Display selected data",
    button_style="primary",
    icon="table"
)

output = widgets.Output()


# --------------------------------------------------
# 3. UPDATE AVAILABLE COLUMNS WHEN DATASET CHANGES
# --------------------------------------------------

def update_columns(change=None):

    selected_dataset = dataset_selector.value
    df = datasets[selected_dataset]

    column_selector.options = list(df.columns)

    # Select the first 8 columns by default.
    column_selector.value = tuple(df.columns[:8])


dataset_selector.observe(update_columns, names="value")
update_columns()


# --------------------------------------------------
# 4. DISPLAY SELECTED DATA
# --------------------------------------------------

def display_selected_data(button):

    with output:
        clear_output(wait=True)

        dataset_name = dataset_selector.value
        df = datasets[dataset_name]

        selected_columns = list(column_selector.value)

        if not selected_columns:
            print("Please select at least one column.")
            return

        # Display dataset information.
        print("=" * 70)
        print(f"DATASET: {dataset_name}")
        print("=" * 70)

        print(f"Total rows:    {len(df):,}")
        print(f"Total columns: {len(df.columns)}")
        print(f"Selected columns: {len(selected_columns)}")

        # Display column information.
        if show_metadata.value:

            metadata = pd.DataFrame({
                "Column": df[selected_columns].columns,
                "Data type": df[selected_columns].dtypes.astype(str).values,
                "Missing values": df[selected_columns].isna().sum().values,
                "Missing (%)": (
                    df[selected_columns].isna().mean().values * 100
                ).round(2)
            })

            print("\nCOLUMN INFORMATION")
            display(metadata)

        # Determine which rows to display.
        start = max(0, start_row.value)

        if show_all_rows.value:
            end = len(df)
        else:
            end = min(start + row_limit.value, len(df))

        # Display the selected columns and rows.
        selected_data = df.iloc[start:end][selected_columns]

        print(f"\nDATA TABLE — Rows {start} to {end - 1}")
        display(selected_data)

        print(f"\nShowing {len(selected_data):,} rows.")


display_button.on_click(display_selected_data)


# --------------------------------------------------
# 5. DISPLAY THE INTERFACE
# --------------------------------------------------

display(
    widgets.VBox([
        widgets.HTML("<h2>UCDP Dataset Explorer</h2>"),
        dataset_selector,
        widgets.HTML(
            "<b>Select the columns you want to display.</b>"
            "<br>Use Ctrl (Windows/Linux) or Cmd (Mac) "
            "to select multiple columns."
        ),
        column_selector,
        row_limit,
        start_row,
        show_all_rows,
        show_metadata,
        display_button,
        output
    ])
)


Loaded: UCDP/PRIO Armed Conflict — 2,816 rows, 28 columns
Loaded: UCDP GED Events — 417,968 rows, 49 columns


In [5]:

for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    overview = pd.DataFrame({
        "Column name": df.columns,
        "Data type": df.dtypes.astype(str).values,
        "Non-null values": df.notna().sum().values,
        "Missing values": df.isna().sum().values,
        "Unique values": df.nunique().values
    })

    display(overview)



UCDP/PRIO Armed Conflict


,Column name,Data type,Non-null values,Missing values,Unique values
0,conflict_id,int64,2816,0,303
1,location,object,2816,0,165
2,side_a,object,2816,0,121
3,side_a_id,object,2816,0,121
4,side_a_2nd,object,479,2337,172
5,side_b,object,2816,0,671
6,side_b_id,object,2816,0,679
7,side_b_2nd,object,147,2669,43
8,incompatibility,int64,2816,0,3
9,territory_name,object,1586,1230,145



UCDP GED Events


,Column name,Data type,Non-null values,Missing values,Unique values
0,id,int64,417968,0,417968
1,relid,object,417968,0,417968
2,year,int64,417968,0,37
3,active_year,bool,417968,0,2
4,code_status,object,417968,0,1
5,type_of_violence,int64,417968,0,3
6,conflict_dset_id,float64,416457,1511,1532
7,conflict_new_id,int64,417968,0,1581
8,conflict_name,object,417968,0,1579
9,dyad_dset_id,float64,417966,2,1804


In [9]:

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output

# ============================================================
# 1. CONFIGURATION AND VALIDATION
# ============================================================

# Columns that should not be treated as analytical variables.
# These are identifiers, coordinates, or source/text metadata.
BLOCKED_COLUMNS = {
    "id", "relid",
    "conflict_id", "conflict_dset_id", "conflict_new_id",
    "dyad_dset_id", "dyad_new_id",
    "side_a_id", "side_a_new_id", "side_a_dset_id",
    "side_b_id", "side_b_new_id", "side_b_dset_id",
    "side_a_2nd", "side_b_2nd",
    "country_id",
    "priogrid_gid",
    "latitude", "longitude",
    "where_coordinates", "geom_wkt",
    "source_office", "source_article",
    "source_headline", "source_original",
    "where_description"
}

# Words indicating that a column is an identifier or location,
# rather than a meaningful quantitative measurement.
BLOCKED_PATTERNS = (
    "_id", "_gid", "coordinates",
    "latitude", "longitude",
    "geom", "source_", "headline",
    "original", "description"
)

def is_blocked_column(column):
    name = str(column).lower()

    if name in BLOCKED_COLUMNS:
        return True

    if any(pattern in name for pattern in BLOCKED_PATTERNS):
        return True

    return False


def get_valid_measures(df):
    """
    Return numerical columns that are suitable for quantitative
    analysis, excluding identifiers and coordinate fields.
    """
    valid = []

    for column in df.columns:
        if is_blocked_column(column):
            continue

        if pd.api.types.is_numeric_dtype(df[column]):
            valid.append(column)

    return valid


def get_valid_categories(df):
    """
    Return columns suitable for grouping the bar chart.
    Exclude IDs, coordinates, and free-text source metadata.
    """
    valid = []

    for column in df.columns:
        if is_blocked_column(column):
            continue

        if (
            pd.api.types.is_numeric_dtype(df[column])
            or pd.api.types.is_object_dtype(df[column])
            or isinstance(df[column].dtype, pd.CategoricalDtype)
            or pd.api.types.is_datetime64_any_dtype(df[column])
            or pd.api.types.is_bool_dtype(df[column])
        ):
            valid.append(column)

    return valid


# ============================================================
# 2. CREATE WIDGETS
# ============================================================

dataset_dropdown = widgets.Dropdown(
    options=list(datasets.keys()),
    description="Dataset:",
    layout=widgets.Layout(width="500px")
)

group_dropdown = widgets.Dropdown(
    description="Group by:",
    layout=widgets.Layout(width="500px")
)

measure_dropdown = widgets.Dropdown(
    description="Measure:",
    layout=widgets.Layout(width="500px")
)

aggregation_dropdown = widgets.Dropdown(
    options=[
        ("Sum", "sum"),
        ("Mean (average)", "mean"),
        ("Median", "median"),
        ("Maximum", "max"),
        ("Minimum", "min"),
        ("Count of records", "count")
    ],
    value="sum",
    description="Aggregate:",
    layout=widgets.Layout(width="500px")
)

sort_dropdown = widgets.Dropdown(
    options=[
        ("Grouping attribute order", "category"),
        ("Ascending (lowest to highest)", "ascending"),
        ("Descending (highest to lowest)", "descending")
    ],
    value="category",
    description="Bar ordering:",
    layout=widgets.Layout(width="500px")
)

top_n_slider = widgets.IntSlider(
    value=20,
    min=5,
    max=417968,
    step=5,
    description="Top groups:",
    continuous_update=False,
    layout=widgets.Layout(width="500px")
)

chart_title_input = widgets.Text(
    value="Custom Data Analysis",
    description="Chart title:",
    layout=widgets.Layout(width="500px")
)

plot_button = widgets.Button(
    description="Generate Bar Chart",
    button_style="success",
    icon="bar-chart"
)

output = widgets.Output()


# ============================================================
# 3. UPDATE AVAILABLE OPTIONS WHEN DATASET CHANGES
# ============================================================

def update_options(change=None):

    dataset_name = dataset_dropdown.value
    df = datasets[dataset_name]

    valid_categories = get_valid_categories(df)
    valid_measures = get_valid_measures(df)

    group_dropdown.options = valid_categories

    # Include a record-count option that works for any dataset.
    measure_options = [
        ("Number of records", "__record_count__")
    ] + [(column, column) for column in valid_measures]

    measure_dropdown.options = measure_options

    # Select sensible defaults where available.
    if "year" in valid_categories:
        group_dropdown.value = "year"
    elif valid_categories:
        group_dropdown.value = valid_categories[0]

    if "best" in valid_measures:
        measure_dropdown.value = "best"
    elif valid_measures:
        measure_dropdown.value = valid_measures[0]
    else:
        measure_dropdown.value = "__record_count__"


dataset_dropdown.observe(update_options, names="value")

update_options()


# ============================================================
# 4. GENERATE BAR CHART
# ============================================================

def generate_chart(button):

    with output:
        clear_output(wait=True)

        dataset_name = dataset_dropdown.value
        df = datasets[dataset_name]

        group_column = group_dropdown.value
        measure_column = measure_dropdown.value
        aggregation = aggregation_dropdown.value
        top_n = top_n_slider.value

        # ----------------------------------------------------
        # Validate selected grouping column
        # ----------------------------------------------------

        if not group_column or group_column not in df.columns:
            print("WARNING: Please select a valid grouping attribute.")
            return

        if is_blocked_column(group_column):
            print(
                f"WARNING: '{group_column}' is an identifier, "
                "coordinate, or metadata field. Choose another "
                "grouping attribute."
            )
            return

        # ----------------------------------------------------
        # Validate selected measurement
        # ----------------------------------------------------

        if measure_column == "__record_count__":
            aggregation = "count"

        elif measure_column not in df.columns:
            print("WARNING: Please select a valid numerical measure.")
            return

        elif is_blocked_column(measure_column):
            print(
                f"WARNING: '{measure_column}' is an identifier, "
                "coordinate, or metadata field. It cannot be used "
                "as a quantitative measure. Choose another attribute."
            )
            return

        elif not pd.api.types.is_numeric_dtype(df[measure_column]):
            print(
                f"WARNING: '{measure_column}' is not numerical. "
                "Choose a numerical measurement instead."
            )
            return

        # ----------------------------------------------------
        # Prepare data
        # ----------------------------------------------------

        if measure_column == "__record_count__":
            chart_data = df[[group_column]].copy()
            chart_data["__value__"] = 1
            value_column = "__value__"
        else:
            chart_data = df[[group_column, measure_column]].copy()
            value_column = measure_column

        chart_data = chart_data.dropna(subset=[group_column])

        # Check that the selected grouping column has usable values.
        if chart_data.empty:
            print("WARNING: No usable data for the selected attribute.")
            return

        # ----------------------------------------------------
        # Aggregate data
        # ----------------------------------------------------

        grouped = chart_data.groupby(
            group_column,
            observed=True
        )[value_column]

        aggregation_functions = {
            "sum": grouped.sum,
            "mean": grouped.mean,
            "median": grouped.median,
            "max": grouped.max,
            "min": grouped.min,
            "count": grouped.count
        }

        result = aggregation_functions[aggregation]()

        result = result.dropna()

        if result.empty:
            print("WARNING: No valid values remain after aggregation.")
            return

        # ----------------------------------------------------
        # CONTROL BAR ORDERING
        # ----------------------------------------------------

        sort_option = sort_dropdown.value

        if sort_option == "ascending":
            result = result.sort_values(ascending=True)

        elif sort_option == "descending":
            result = result.sort_values(ascending=False)

        elif sort_option == "category":
            # Sort by the grouping attribute, not by bar height.
            if pd.api.types.is_numeric_dtype(df[group_column]):
                result = result.sort_index(ascending=True)
            else:
                result = result.sort_index(
                    key=lambda index: index.astype(str).str.lower(),
                    ascending=True
                )

        # Limit the number of displayed groups.
        # For ascending/descending, retain the groups at the
        # selected end of the ranking.
        if sort_option == "ascending":
            result = result.head(top_n)

        elif sort_option == "descending":
            result = result.head(top_n)

        else:
            result = result.head(top_n)

        # ----------------------------------------------------
        # Display summary information
        # ----------------------------------------------------

        print(f"Dataset: {dataset_name}")
        print(f"Grouping attribute: {group_column}")
        print(f"Measurement: {measure_column}")
        print(f"Aggregation: {aggregation}")
        print(f"Groups displayed: {len(result)}")
        print(f"Source records: {len(df):,}")

        print("\nAggregated data:")
        display(result.rename("Value").to_frame())

        # ----------------------------------------------------
        # Plot bar chart
        # ----------------------------------------------------

        plt.figure(figsize=(12, 6))

        result.plot(
            kind="bar",
            color="steelblue",
            edgecolor="black"
        )

        plt.title(chart_title_input.value, fontsize=15)
        plt.xlabel(group_column, fontsize=12)
        plt.ylabel(
            "Number of records" if aggregation == "count"
            else f"{aggregation.title()} of {measure_column}",
            fontsize=12
        )

        plt.xticks(rotation=45, ha="right")
        plt.grid(axis="y", linestyle="--", alpha=0.4)

        plt.tight_layout()
        plt.show()


plot_button.on_click(generate_chart)


# ============================================================
# 5. DISPLAY THE INTERACTIVE INTERFACE
# ============================================================

display(
    widgets.VBox([
        widgets.HTML("<h3>Custom Bar Chart Generator</h3>"),
        dataset_dropdown,
        group_dropdown,
        measure_dropdown,
        aggregation_dropdown,
        sort_dropdown,
        top_n_slider,
        chart_title_input,
        plot_button,
        output
    ])
)
